## BERT 파인튜닝

### ✅ `distilbert-base-uncased` 모델 정보

| 항목                     | `distilbert-base-uncased`          |
|------------------------|----------------------------------|
| 🔢 모델 타입               | Encoder (BERT 계열)                  |
| 📦 사전학습 목적             | 마스킹된 언어 모델 (Masked LM)         |
| 🧠 파라미터 수              | 약 66M                              |
| 🏗️ 레이어 수              | 6                                  |
| 🧩 히든 크기 (Hidden Size) | 768                                |
| 🔁 어텐션 헤드 수            | 12                                 |
| 📏 최대 시퀀스 길이           | 512 tokens                         |
| 📄 토크나이저 타입            | WordPiece (BERT형)                  |
| 💬 생성 능력               | ❌ (비생성형, 분류에 적합)              |
| ⚙️ 사용 예시               | 텍스트 분류, 감정 분석 등               |
| ⚡ 연산 효율성               | 빠름 (BERT의 경량화 버전)              |


# 📌 자연어 처리 텍스트 분류 파이프라인 전체 흐름 이해하기

이 코드는 영화 리뷰 텍스트가 **긍정인지 부정인지 분류하는 자연어 처리(NLP)** 예제입니다.  

---

## ✅ 단계 1: 문제 정의 - 텍스트를 분류하는 모델 만들기
- 목적: 영화 리뷰를 보고 **긍정(좋다)**인지 **부정(싫다)**인지 자동으로 판단하는 모델을 만든다.
- 입력: 영화 리뷰 텍스트 (예: `"This movie was awesome!"`)
- 출력: 0 (부정) 또는 1 (긍정)

> 🎯 예를 들면, 모델에게 `"It was amazing!"`이라는 문장을 주면 → `1(긍정)`이라고 판단하게 만드는 것이 목표입니다.

---

## ✅ 단계 2: 데이터 준비 - 영화 리뷰 데이터를 가져오기
- 사용하는 데이터셋: **IMDB 영화 리뷰 데이터셋**
- 예제에서는 **20개 리뷰만 샘플로 사용**하고, 이 중 80%는 학습용, 20%는 테스트용으로 나눕니다.

> 🧩 왜 데이터를 나눌까?
> - **학습 데이터**로 모델을 훈련시키고  
> - **테스트 데이터**로 모델이 잘 학습되었는지 검증합니다.

---

## ✅ 단계 3: 텍스트 전처리 - 단어를 숫자로 바꾸기
- 컴퓨터는 글자가 아니라 숫자를 이해합니다.
- 그래서 `"I loved the movie"` 같은 문장을 숫자 토큰으로 바꿔야 합니다.
- 이 작업을 도와주는 것이 바로 **토크나이저(Tokenzier)** 입니다.

> ✨ 비유하자면, 문장을 ‘레고 블록’처럼 잘게 쪼개고, 각각에 번호를 붙이는 과정입니다.

---

## ✅ 단계 4: 모델 준비 - 사전학습된 똑똑한 모델 불러오기
- 우리는 처음부터 모델을 만드는 게 아니라, 이미 영어를 어느 정도 학습해둔 모델(**DistilBERT**)을 가져옵니다.
- 이 모델에 "긍정/부정 분류"라는 **새로운 임무를 학습시키는 것**입니다. 이를 **파인튜닝(Fine-tuning)**이라고 합니다.

> 🧠 쉽게 말해, "기초 영어는 이미 배운 친구에게 감정 분석이라는 새 과목을 가르치는 것"입니다.

---

## ✅ 단계 5: 모델 훈련 - 데이터로 학습시키기
- 앞에서 준비한 텍스트(숫자 형태)와 라벨(긍정/부정)을 가지고 모델을 학습합니다.
- 학습은 30번 반복해서 진행됩니다 (30 에폭).

> 🚀 학습이란? 모델이 "이 문장이 긍정인지 부정인지" 스스로 맞추도록 훈련시키는 과정입니다.

---

## ✅ 단계 6: 예측하기 - 새로운 문장에 대해 모델이 판단하도록 하기
- 학습이 끝난 모델에 새로운 문장을 넣어보면, 긍정인지 부정인지 판단해줍니다.
- 예: `"It was great to see John Ritter"` → 모델이 **긍정**이라고 판단하면 `1`을 반환합니다.




In [ ]:
##세션 다시 시작 필요
!pip install -q transformers datasets
# 🤖 Hugging Face 라이브러리 설치
# transformers: 사전학습된 NLP 모델을 쉽게 사용할 수 있게 해주는 라이브러리
# datasets: 다양한 자연어처리용 데이터셋을 손쉽게 불러올 수 있는 라이브러리
!pip install -U "datasets<=2.18.0" "fsspec<=2023.6.0"


In [ ]:
# ✅ 1. 데이터 로드
from datasets import load_dataset
dataset = load_dataset("imdb", split="train[:50]").train_test_split(test_size=0.2)
# 🔍 포인트: IMDB 영화 리뷰 데이터셋 중 50개 샘플만 가져와서 학습용/평가용으로 8:2 비율로 나눕니다
# - 'train[:50]': 전체 학습 데이터 중 앞에서 50개만 사용 (예제 실행용으로 작게 설정한 것)
# - train_test_split: 데이터를 학습용(train)과 테스트용(test)으로 자동 분리



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
sample = dataset["train"][7]
print(f"📝 리뷰 내용:\n{sample['text']}\n")
print(f"🏷️ 라벨 (0=부정, 1=긍정): {sample['label']}")

📝 리뷰 내용:
I received this movie as a gift, I knew from the DVD cover, this movie are going to be bad.After not watching it for more than a year I finally watched it. what a pathetic movie.<br /><br />I almost didn't finish watching this bad movie,but it will be unfair of me to write a review without watching the complete movie.<br /><br />Trust me when I say " this movie sucks" I am truly shocked that some bad filmmaker wane bee got even financed to make this pathetic movie, But it couldn't have cost more than $20 000 to produce this movie. all you need are a cheap camcorder or a cell phone camera .about 15 people with no acting skills, a scrip that were written by a couple of drunk people.<br /><br />In the fist part of this ultra bad move a reporter (Tara Woodley )run a suppose to be drunk man over on her way to report on a hunted town. He are completely unharmed. They went to a supposed to be abandon house ,but luckily for the it almost complete furnished and a bottle of liquor on t

In [ ]:
 # ✅ 2. 모델 및 토크나이저 준비
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# 🔍 포인트: 토크나이저란? → 텍스트를 숫자 토큰으로 바꾸는 역할
# "나는 배가 고프다" → [101, 1432, 2234, 1678, 102] ← 이런 식으로 숫자로 바꿔줘야 모델이 이해 가능

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")
# 🔍 distilbert-base-uncased: BERT를 경량화한 모델 (빠르고 정확도도 준수)
# 포인트: 사전학습(pretrained)된 모델이므로, 적은 데이터로도 빠르게 학습 가능

# ✅ 3. 데이터 전처리 (텍스트를 숫자 토큰으로 변환)
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding=True)
# 🔍 포인트: tokenizer 함수로 모든 리뷰 텍스트를 숫자 토큰으로 바꿉니다
# - truncation=True: 문장이 너무 길면 자르고
# - padding=True: 짧은 문장은 0으로 채워 길이를 맞춰줍니다

dataset = dataset.map(tokenize, batched=True)
# 🔍 map 함수: 전체 데이터셋에 대해 위 tokenize 함수를 한꺼번에 적용
# - batched=True: 여러 데이터를 동시에 처리해서 속도 향상


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
# ✅ 4. 훈련 설정 및 Trainer 구성
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

# 정확도 계산 함수
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

args = TrainingArguments(
    output_dir="test",  # 모델이 저장될 경로
    per_device_train_batch_size=8,  # 학습 시 배치 크기 (GPU 하나당 8개씩 처리)
    num_train_epochs=15,  # 에폭 수 (데이터셋을 15번 반복해서 학습)
     report_to = "none",  # wandb, tensorboard 등 외부 로깅 툴 비활성화
        logging_steps=1                   # 💡 몇 step마다 로그 출력할지
)

trainer = Trainer(
    model=model,  # 사용할 모델
    args=args,  # 훈련 설정값
    train_dataset=dataset["train"],  # 학습용 데이터셋
    eval_dataset=dataset["test"],  # 검증용 데이터셋
    compute_metrics=compute_metrics  # 💡 정확도 계산 함수 추가

)
# 🔍 포인트: Trainer는 Hugging Face에서 제공하는 편리한 훈련 도구
# 복잡한 학습 코드 없이도 손쉽게 훈련 가능

In [ ]:
# ✅ 5. 파인튜닝 실행 (모델 학습)
trainer.train()
# 🔍 포인트: 위 한 줄만으로 모델이 훈련됨! (사전학습된 모델을 우리가 준비한 데이터에 맞춰 미세조정)

Step,Training Loss
1,0.774700
2,0.597000
3,0.438800
4,0.328500
5,0.238700
6,0.156400
7,0.119300
8,0.091100
9,0.052000
10,0.041500


TrainOutput(global_step=75, training_loss=0.04163592360758533, metrics={'train_runtime': 37.1019, 'train_samples_per_second': 16.172, 'train_steps_per_second': 2.021, 'total_flos': 79480439193600.0, 'train_loss': 0.04163592360758533, 'epoch': 15.0})

In [ ]:
results = trainer.evaluate()
print(results)


{'eval_loss': 0.001097958185710013, 'eval_accuracy': 1.0, 'eval_runtime': 0.2316, 'eval_samples_per_second': 43.174, 'eval_steps_per_second': 8.635, 'epoch': 15.0}


In [ ]:
# ✅ 학습된 모델로 실제 예측 수행
text = "I would put this at the top of my list of films in the category of unwatchable trash!"
inputs = tokenizer(text, return_tensors="pt").to("cuda")
# 🔍 포인트: 새 문장을 토큰화한 후, GPU로 보내줌 (".to('cuda')" 필수)
# "pt": PyTorch 텐서 형식으로 변환

output = model(**inputs)
label = output.logits.argmax(-1).item()
# 🔍 포인트: 모델 출력(logits)은 [부정 점수, 긍정 점수] 형태의 벡터
# argmax(-1): 가장 높은 점수의 인덱스 (0 또는 1)를 예측값으로 사용

print("긍정" if label == 1 else "부정")
# 🔍 최종 결과 출력: 긍정인지 부정인지 해석해서 보여줌

부정
